This notebook will pre-process the data and also carry out feature engineering

In [167]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, isnan, count, lit
from pyspark.sql.types import IntegerType, DoubleType, StringType, StructType, StructField

In [168]:
# Importing Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [169]:
# Create Spark session
spark = SparkSession.builder \
    .appName("Customer Churn Prediction") \
    .getOrCreate()

In [170]:
# Load Data
data_pp = spark.read.csv('../data/cleaned/cleaned_data.csv', header=True, inferSchema=True)

In [171]:
data_pp.show(5)

+----------+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+--------------+------------+-----+
|customerID|gender|SeniorCitizen|Partner|Dependents|tenure|PhoneService|   MultipleLines|InternetService|OnlineSecurity|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingMovies|      Contract|PaperlessBilling|       PaymentMethod|MonthlyCharges|TotalCharges|Churn|
+----------+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+--------------+------------+-----+
|7590-VHVEG|Female|            0|    Yes|        No|     1|          No|No phone service|            DSL|            No|         Yes|              No|         No|    

In [172]:
data_pp.schema

StructType([StructField('customerID', StringType(), True), StructField('gender', StringType(), True), StructField('SeniorCitizen', IntegerType(), True), StructField('Partner', StringType(), True), StructField('Dependents', StringType(), True), StructField('tenure', IntegerType(), True), StructField('PhoneService', StringType(), True), StructField('MultipleLines', StringType(), True), StructField('InternetService', StringType(), True), StructField('OnlineSecurity', StringType(), True), StructField('OnlineBackup', StringType(), True), StructField('DeviceProtection', StringType(), True), StructField('TechSupport', StringType(), True), StructField('StreamingTV', StringType(), True), StructField('StreamingMovies', StringType(), True), StructField('Contract', StringType(), True), StructField('PaperlessBilling', StringType(), True), StructField('PaymentMethod', StringType(), True), StructField('MonthlyCharges', DoubleType(), True), StructField('TotalCharges', DoubleType(), True), StructField(

In [173]:
data_pp.select('MultipleLines').distinct().show()

+----------------+
|   MultipleLines|
+----------------+
|No phone service|
|              No|
|             Yes|
+----------------+



In [174]:
schema_table = pd.DataFrame(data_pp.dtypes, columns=["Column", "DataType"])
schema_table

,Column,DataType
0,customerID,string
1,gender,string
2,SeniorCitizen,int
3,Partner,string
4,Dependents,string
5,tenure,int
6,PhoneService,string
7,MultipleLines,string
8,InternetService,string
9,OnlineSecurity,string


In [175]:
# Data Type Conversion

data_pp = data_pp.withColumn("SeniorCitizen", col("SeniorCitizen").cast(IntegerType()))
data_pp = data_pp.withColumn("tenure", col("tenure").cast(IntegerType()))
data_pp = data_pp.withColumn("MonthlyCharges", col("MonthlyCharges").cast(DoubleType()))
data_pp = data_pp.withColumn("TotalCharges", col("TotalCharges").cast(DoubleType()))

In [176]:
data_pp = data_pp.drop('CustomerID')

In [177]:
data_pp.select('PaymentMethod').distinct().show()

+--------------------+
|       PaymentMethod|
+--------------------+
|Credit card (auto...|
|        Mailed check|
|Bank transfer (au...|
|    Electronic check|
+--------------------+



In [178]:
# Encoding Target Variables
data_pp = data_pp.withColumn("label", when(col("Churn") == "Yes", 1).otherwise(0))
data_pp = data_pp.drop('Churn')

### Feature Engineering

In [179]:
numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
cat_features = [col for col in data_pp.columns if col not in numeric_features + ['label']]

In [180]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder

# String Indexing and One-Hot Encoding
indexers = [StringIndexer(inputCol=c, outputCol=c+"_Index", handleInvalid="keep")
            for c in cat_features]

encoders = [OneHotEncoder(inputCol=c+"_Index", outputCol=c+"_OHE") for c in cat_features]

In [181]:
# Into a single vector

from pyspark.ml.feature import VectorAssembler

feature_cols = numeric_features + [c+"_OHE" for c in cat_features]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

In [182]:
# Pipeline
from pyspark.ml import Pipeline

pipeline = Pipeline(stages=indexers + encoders + [assembler])

In [183]:
# Fit and Transform

data_transformed = pipeline.fit(data_pp).transform(data_pp)
data_transformed.select("features", "label").show(5, truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------+-----+
|features                                                                                                                                    |label|
+--------------------------------------------------------------------------------------------------------------------------------------------+-----+
|(46,[0,1,2,4,5,8,9,12,15,17,19,23,25,28,31,34,37,40,42],[1.0,29.85,29.85,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])  |0    |
|(46,[0,1,2,3,5,7,9,11,13,17,20,22,26,28,31,34,39,41,43],[34.0,56.95,1889.5,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0])|0    |
|(46,[0,1,2,3,5,7,9,11,13,17,20,23,25,28,31,34,37,40,43],[2.0,53.85,108.15,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0]) |1    |
|(46,[0,1,2,3,5,7,9,12,15,17,20,22,26,29,31,34,39,41,44],[45.0,42.3,1840.75,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.

In [184]:
# Train-Test Split

train_data, test_data = data_transformed.randomSplit([0.8, 0.2], seed=42)